In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Labs"
)

xpt_file = DATA_DIR / "L10_C.xpt"
csv_file = DATA_DIR / "L10_C_corrected.csv"


# ============================================================
# 2. READ XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

print("=" * 70)
print("NHANES 2003-2004 L10_C XPT -> CSV")
print("=" * 70)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Variables:", df.columns.tolist())


# ============================================================
# 3. KNOWN XPT TINY VALUE
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


# ============================================================
# 4. CHECK BEFORE ANY CHANGES
# ============================================================

tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)

total_tiny = int(
    tiny_counts.sum()
)

total_zeros = int(
    zero_counts.sum()
)


print("\n--- ORIGINAL XPT CHECK ---")

print(
    "True numeric zeros:",
    total_zeros
)

print(
    "Tiny-value occurrences:",
    total_tiny
)


if total_tiny > 0:

    print("\nVariables containing tiny value:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )


# ============================================================
# 5. SEQN FIDELITY
# ============================================================

print("\n--- SEQN CHECK ---")

if "SEQN" not in df.columns:

    raise ValueError(
        "SEQN not found."
    )


print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)


seqn_values = (
    df["SEQN"]
    .dropna()
)


# SEQN must contain whole-number identifiers

if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values)
    )
):

    raise ValueError(
        "SEQN contains unexpected decimal values."
    )


# Export SEQN without .0

df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print(
    "SEQN dtype after formatting:",
    df["SEQN"].dtype
)


# ============================================================
# 6. LBXGH FIDELITY
# ============================================================

print("\n--- LBXGH CHECK ---")

if "LBXGH" not in df.columns:

    raise ValueError(
        "LBXGH not found."
    )


lbxgh = df["LBXGH"]


print(
    "Nonmissing:",
    int(lbxgh.notna().sum())
)

print(
    "Missing:",
    int(lbxgh.isna().sum())
)

print(
    "Minimum:",
    lbxgh.min()
)

print(
    "Maximum:",
    lbxgh.max()
)


# ============================================================
# 7. CDC CODEBOOK CHECK
# ============================================================

#
# CDC L10_C:
#
# LBXGH valid range = 3.3 to 18
# Valid observations = 6601
# Missing = 389
#

EXPECTED_ROWS = 6990
EXPECTED_VALID = 6601
EXPECTED_MISSING = 389

MIN_VALID = 3.3
MAX_VALID = 18.0


print("\n--- CDC CODEBOOK VALIDATION ---")


# Row count

if len(df) == EXPECTED_ROWS:

    print(
        "PASS: Row count = 6,990"
    )

else:

    print(
        "FAIL: Row count =",
        len(df)
    )


# Valid LBXGH count

valid_count = int(
    lbxgh.notna().sum()
)

if valid_count == EXPECTED_VALID:

    print(
        "PASS: LBXGH valid count = 6,601"
    )

else:

    print(
        "FAIL: LBXGH valid count =",
        valid_count
    )


# Missing count

missing_count = int(
    lbxgh.isna().sum()
)

if missing_count == EXPECTED_MISSING:

    print(
        "PASS: LBXGH missing = 389"
    )

else:

    print(
        "FAIL: LBXGH missing =",
        missing_count
    )


# Range check

out_of_range = (
    lbxgh.notna()
    &
    (
        (lbxgh < MIN_VALID)
        |
        (lbxgh > MAX_VALID)
    )
)


out_of_range_count = int(
    out_of_range.sum()
)


if out_of_range_count == 0:

    print(
        "PASS: All LBXGH values "
        "are within 3.3–18.0"
    )

else:

    print(
        "FAIL:",
        out_of_range_count,
        "LBXGH values outside codebook range"
    )

    print(
        df.loc[
            out_of_range,
            ["SEQN", "LBXGH"]
        ]
    )


# ============================================================
# 8. DECIMAL PRECISION CHECK
# ============================================================

#
# LBXGH is a measurement.
# DO NOT convert it to integer.
#
# The attached XPT contains measurements
# at one decimal place.
#

nonmissing_lbxgh = (
    lbxgh.dropna()
)


one_decimal = np.all(
    np.isclose(
        nonmissing_lbxgh * 10,
        np.round(
            nonmissing_lbxgh * 10
        )
    )
)


if one_decimal:

    print(
        "PASS: LBXGH values preserve "
        "one-decimal measurement precision."
    )

else:

    print(
        "CHECK: LBXGH contains values "
        "with more than one decimal place."
    )


# ============================================================
# 9. IMPORTANT TINY-VALUE RULE
# ============================================================

#
# DO NOT automatically change tiny values to zero here.
#
# CDC says valid LBXGH begins at 3.3.
# Therefore 0 is NOT a valid LBXGH measurement.
#

lbxgh_tiny = int(
    df["LBXGH"]
    .eq(TINY_VALUE)
    .sum()
)


if lbxgh_tiny == 0:

    print(
        "PASS: No tiny-value artifact "
        "found in LBXGH."
    )

else:

    raise ValueError(
        f"FIDELITY CHECK FAILED: "
        f"LBXGH contains {lbxgh_tiny} "
        f"tiny-value records. "
        f"Do NOT automatically convert them to zero."
    )


# ============================================================
# 10. DESCRIPTIVE STATISTICS
# ============================================================

print("\n--- LBXGH DESCRIPTIVE STATISTICS ---")

print(
    df["LBXGH"]
    .describe()
)


# ============================================================
# 11. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 12. ROUND-TRIP CSV VALIDATION
# ============================================================

check_df = pd.read_csv(
    csv_file,
    low_memory=False
)


print("\n--- CSV ROUND-TRIP CHECK ---")


# SEQN should be clean identifier

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    header = f.readline()

    data_lines = [
        f.readline()
        for _ in range(20)
    ]


seqn_dot_zero = any(
    line.split(",")[0].endswith(".0")
    for line in data_lines
)


if not seqn_dot_zero:

    print(
        "PASS: SEQN exported without .0"
    )

else:

    print(
        "FAIL: SEQN contains .0"
    )


# Compare LBXGH after CSV round trip

original = (
    df["LBXGH"]
    .to_numpy()
)

exported = (
    check_df["LBXGH"]
    .to_numpy()
)


same_values = np.allclose(
    original,
    exported,
    equal_nan=True
)


if same_values:

    print(
        "PASS: LBXGH values preserved "
        "through CSV export."
    )

else:

    print(
        "FAIL: LBXGH values changed "
        "during CSV export."
    )


# Final check

print("\n" + "=" * 70)
print("L10_C CONVERSION COMPLETE")
print("=" * 70)

NHANES 2003-2004 L10_C XPT -> CSV
Rows: 6,990
Columns: 2
Variables: ['SEQN', 'LBXGH']

--- ORIGINAL XPT CHECK ---
True numeric zeros: 0
Tiny-value occurrences: 0

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
SEQN dtype after formatting: Int64

--- LBXGH CHECK ---
Nonmissing: 6601
Missing: 389
Minimum: 3.3
Maximum: 18.0

--- CDC CODEBOOK VALIDATION ---
PASS: Row count = 6,990
PASS: LBXGH valid count = 6,601
PASS: LBXGH missing = 389
PASS: All LBXGH values are within 3.3–18.0
PASS: LBXGH values preserve one-decimal measurement precision.
PASS: No tiny-value artifact found in LBXGH.

--- LBXGH DESCRIPTIVE STATISTICS ---
count    6601.000000
mean        5.487729
std         0.853890
min         3.300000
25%         5.100000
50%         5.300000
75%         5.600000
max        18.000000
Name: LBXGH, dtype: float64

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\2003_2004\Labs\L10_C_corrected.csv

--- CSV ROUN